## Tune the decision threshold for the best XGBoost model

#### Import packages

In [ ]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

#### Load the engineered dataset

In [ ]:
data = pd.read_parquet(
    "../data/processed/train_transaction_features.parquet",
    engine="fastparquet"
)

data.shape

#### Prepare features and target

In [ ]:
y = data["isFraud"]

X = data.drop(columns=["isFraud", "TransactionID"])
X = X.select_dtypes(include=["number"])

print("Number of features:", X.shape[1])

#### Create chronological train-validation split

In [ ]:
split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index]
X_validation = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_validation = y.iloc[split_index:]

print("Training rows:", len(X_train))
print("Validation rows:", len(X_validation))

#### Calculate fraud class weight

In [ ]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Scale pos weight:", scale_pos_weight)

#### Train the best model from notebook: 09_xgboost_hyperparameter_tuning

In [ ]:
# Best hyperparameter combination from notebook: 09_xgboost_hyperparameter_tuning
best_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)

#### Generate fraud probabilities

In [ ]:
y_probability = best_model.predict_proba(X_validation)[:, 1]

#### Test many thresholds

In [ ]:
thresholds = np.arange(0.05, 0.96, 0.025)

results = []

for threshold in thresholds:
    y_pred_threshold = (
        y_probability >= threshold
    ).astype(int)

    # Extract confusion matrix values
    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        y_pred_threshold
    ).ravel()

    # Calculate fraud-class metrics
    precision = precision_score(
        y_validation,
        y_pred_threshold,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        y_pred_threshold
    )

    f1 = f1_score(
        y_validation,
        y_pred_threshold
    )

    false_positive_rate = fp / (fp + tn)

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "false_positive_rate": false_positive_rate,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn
    })

threshold_results = pd.DataFrame(results)

threshold_results

#### Find the best threshold by F1 score

In [ ]:
threshold_results.sort_values(
    by="f1_score",
    ascending=False
).head(10)

#### Find thresholds with high fraud recall

In [ ]:
high_recall_thresholds = threshold_results[
    threshold_results["recall"] >= 0.80
]

high_recall_thresholds.sort_values(
    by="false_positive_rate",
    ascending=True
).head(10)

#### Find thresholds with false-positive rate under 10%

In [ ]:
# Find thresholds that keep false-positive rate below 10%
low_fp_thresholds = threshold_results[
    threshold_results["false_positive_rate"] < 0.10
]

low_fp_thresholds.sort_values(
    by="recall",
    ascending=False
).head(10)

#### Look for a more aggressive threshold:

In [ ]:
business_thresholds = threshold_results[
    (threshold_results["recall"] >= 0.80) &
    (threshold_results["false_positive_rate"] <= 0.15)
]

business_thresholds.sort_values(
    by="f1_score",
    ascending=False
)

#### Comparison table

In [ ]:
threshold_results[
    threshold_results["threshold"].between(0.30, 0.425)
]
[
    [
        "threshold",
        "precision",
        "recall",
        "f1_score",
        "false_positive_rate",
        "true_positive",
        "false_positive",
        "false_negative",
        "true_negative"
    ]
]

#### Calculate ROC-AUC and PR-AUC

In [ ]:
roc_auc = roc_auc_score(
    y_validation,
    y_probability
)

pr_auc = average_precision_score(
    y_validation,
    y_probability
)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")

#### Plot threshold tradeoffs

In [ ]:
import matplotlib.pyplot as plt

# Plot how precision, recall, and F1 change as threshold changes
plt.figure(figsize=(10, 6))

plt.plot(
    threshold_results["threshold"],
    threshold_results["precision"],
    label="Precision"
)

plt.plot(
    threshold_results["threshold"],
    threshold_results["recall"],
    label="Recall"
)

plt.plot(
    threshold_results["threshold"],
    threshold_results["f1_score"],
    label="F1 Score"
)

plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Precision, Recall, and F1 Across Decision Thresholds")
plt.legend()
plt.grid()
plt.show()

#### Plot false-positive rate

In [ ]:
# Show how raising the threshold changes false-positive rate
plt.figure(figsize=(10, 6))

plt.plot(
    threshold_results["threshold"],
    threshold_results["false_positive_rate"]
)

plt.xlabel("Decision Threshold")
plt.ylabel("False-Positive Rate")
plt.title("False-Positive Rate Across Decision Thresholds")
plt.grid()
plt.show()

#### Save threshold results

In [ ]:
threshold_results.to_csv(
    "../data/processed/best_model_threshold_results.csv",
    index=False
)

## Best Model Threshold Tuning Summary

The tuned `deeper_trees` XGBoost model was evaluated across multiple decision thresholds from 0.05 to 0.95.

The purpose of this analysis was to determine how changing the classification threshold affects fraud recall, precision, F1 score, and false-positive rate.

A lower threshold classifies more transactions as fraud, which usually increases recall but also increases false positives. A higher threshold makes the model more selective, which usually improves precision and reduces false positives but causes more fraud cases to be missed.

### Model Ranking Performance

* ROC-AUC: 0.907
* PR-AUC: 0.518

These metrics evaluate the model across different thresholds rather than at one fixed cutoff. The tuned XGBoost model improved both ranking metrics compared with the earlier baseline models.

### Strict False-Positive-Rate Threshold

A stricter operating strategy was evaluated by requiring the false-positive rate to remain below 10%.

The best threshold satisfying this requirement was 0.425:

* Precision: 0.219
* Recall: 0.731
* F1 score: 0.337
* False-positive rate: 9.31%
* True positives: 2,972
* False positives: 10,616
* False negatives: 1,092
* True negatives: 103,428

At this threshold, the model detected approximately 73.1% of fraudulent transactions while keeping the false-positive rate below 10%.

This threshold may be appropriate when reducing false alerts and analyst workload is a high priority.

### Business-Oriented Threshold

A second threshold strategy was evaluated to prioritize higher fraud detection while allowing a moderately higher false-positive rate.

The business objective was defined as:

* Fraud recall of at least 80%
* False-positive rate below 15%

The selected threshold was 0.325:

* Precision: 0.163
* Recall: 0.804
* F1 score: 0.270
* False-positive rate: 14.77%
* True positives: 3,269
* False positives: 16,845
* False negatives: 795
* True negatives: 97,199

At this threshold, the model detected approximately 80.4% of fraudulent transactions while keeping the false-positive rate below 15%.

Lowering the threshold to 0.300 increased recall to 82.2%, but the false-positive rate increased to 16.7%.

Increasing the threshold to 0.350 reduced the false-positive rate to 13.1%, but recall decreased to 78.4%, falling below the selected 80% fraud-detection target.

Therefore, 0.325 provides the strongest balance for the chosen business objective.

### Final Threshold Decision

The preferred production threshold for this project is 0.325.

This threshold was selected because it:

* Detects over 80% of fraudulent transactions
* Keeps the false-positive rate below 15%
* Provides a clear balance between fraud detection and operational review workload
* Supports a more aggressive fraud-detection strategy than the strict 10% false-positive-rate threshold

The threshold can still be adjusted depending on business priorities. For example, a financial institution that prioritizes fraud prevention may choose a lower threshold, while an organization with limited analyst review capacity may choose a higher threshold.
